# Практика 1. Объекты, имена, доступ

Теория: блоки 0–2.

Девять задач на то, где хранится имя, что происходит при присваивании и как ищется
атрибут.

**Как устроено.** Условие → ячейка → проверка. Ячейки, которые надо писать самому, помечены
комментарием в первой строке. Если что-то пошло не так — перезапусти ядро и выполни
служебную ячейку заново.

In [13]:
# Служебное. Запусти эту ячейку один раз перед задачами.
def check(name, fn):
    """Запускает проверку и печатает результат."""
    try:
        fn()
        print("✓", name)
    except AssertionError as e:
        print("✗", name, "—", e)
    except Exception as e:
        print("✗", name, "—", type(e).__name__, e)


def eq(got, want):
    assert got == want, f"получено {got!r}, ожидалось {want!r}"


def expect(exc, fn):
    try:
        fn()
    except exc:
        return
    raise AssertionError(f"ожидалось {exc.__name__}")

## Задачи

### Задача 1. Мутабельный аргумент по умолчанию

Ниже три вызова одной функции. Запиши три строки вывода **до** запуска, потом сверь.

In [1]:
def add(item, box=[]):
    box.append(item)
    return box

print(add("a"))
print(add("b"))
print(add("c", []))

['a']
['a', 'b']
['c']


### Задача 2. `+=` внутри кортежа

Кортеж неизменяем, но внутри него список. Что напечатается и почему ошибка не отменяет
изменение?

In [3]:
t = (1, [2])
u = t
try:
    t[1] += [3]
except TypeError as e:
    print("ошибка:", type(e).__name__)
print(t, u, t is u)

(1, [2, 4]) (1, [2, 4]) True


### Задача 3. `+=` для списка в атрибуте класса

Класс один, экземпляров два, список объявлен в классе. Что напечатается и появится ли
`items` в словаре экземпляра?

In [7]:
class C:
    items = []

c1, c2 = C(), C()
print(c1.__dict__)
c1.items += "a"
print(c2.items, C.items)
print("items" in c1.__dict__)
print(c1.__dict__)

{}
['a'] ['a']
False
{}


### Задача 4. Корзина, которая одна на всех

Две корзины делят один список товаров. Починить так, чтобы у каждой был свой, не меняя
`add`.

In [12]:
# СЛОМАНО. Правь эту ячейку, пока проверка не станет зелёной.
class Cart:
    def __init__(self):
        self.items = []

    def add(self, name):
        self.items.append(name)


a, b = Cart(), Cart()
a.add("хлеб")
print(b.items)

[]


In [15]:
# Проверка. Запускай после того, как напишешь решение.
a, b = Cart(), Cart()
a.add("хлеб")
check("у второй корзины пусто", lambda: eq(b.items, []))
check("у первой появился товар", lambda: eq(a.items, ["хлеб"]))
check("список живёт в экземпляре", lambda: eq("items" in a.__dict__, True))

✓ у второй корзины пусто
✓ у первой появился товар
✓ список живёт в экземпляре


### Задача 5. Функция, которая портит свой аргумент

`add_tag` меняет переданный список. Переписать так, чтобы возвращался новый список,
а вход оставался нетронутым. Сигнатура та же.

In [19]:
# Правь эту ячейку.
def add_tag(tags, tag):
    return tags + [tag]

In [20]:
# Проверка. Запускай после того, как напишешь решение.
src = ["a"]
out = add_tag(src, "b")
check("вход не изменился", lambda: eq(src, ["a"]))
check("результат содержит тег", lambda: eq(out, ["a", "b"]))
check("результат — новый список", lambda: eq(out is src, False))

✓ вход не изменился
✓ результат содержит тег
✓ результат — новый список


### Задача 6. Слияние настроек

**Контракт.** `merge_settings(base, override)` возвращает **новый** словарь:

- ключи из `override` побеждают;
- если по одному ключу в обоих словарях лежит словарь, они сливаются на один уровень;
- `base` и `override` после вызова не меняются;
- вложенные словари глубже одного уровня копировать не нужно.

Имена и сигнатура даны, тела — твои.

In [44]:
# ↓ твой код ↓
def merge_settings(base, override, with_inner=True):
    print(base, override)
    unique_keys = set(base.keys()) | set(override.keys())
    merged = {}
    
    for key in unique_keys:
        if key in base and key in override:
            if isinstance(base[key], dict) and isinstance(override[key], dict) and with_inner:
                merged[key] = merge_settings(base[key], override[key], False)
            else:
                merged[key] = override[key]
        elif key in override:
            merged[key] = override[key]
        else:
            merged[key] = base[key]

    return merged

In [45]:
# Проверка. Запускай после того, как напишешь решение.
base = {"db": {"host": "localhost", "port": 5432}, "debug": False}
over = {"db": {"port": 6543}, "debug": True}
merged = merge_settings(base, over)

check("верхний уровень слит", lambda: eq(merged["debug"], True))
check("вложенный словарь слит", lambda: eq(merged["db"], {"host": "localhost", "port": 6543}))
check("base не изменён", lambda: eq(base["db"], {"host": "localhost", "port": 5432}))
check("override не изменён", lambda: eq(over["db"], {"port": 6543}))
check("результат — новый словарь", lambda: eq(merged is base, False))

{'db': {'host': 'localhost', 'port': 5432}, 'debug': False} {'db': {'port': 6543}, 'debug': True}
{'host': 'localhost', 'port': 5432} {'port': 6543}
✓ верхний уровень слит
✓ вложенный словарь слит
✓ base не изменён
✓ override не изменён
✓ результат — новый словарь


### Задача 7. Объект, который помнит свои присваивания

**Контракт.** `Tracked` записывает имена всех присвоенных ему атрибутов:

- `history` — список имён в порядке присваивания;
- повторное присваивание того же имени попадает в историю снова;
- само имя `history` в историю не попадает;
- значения при этом должны читаться как обычно.

Подсказка: без `object.__setattr__` здесь получится рекурсия.

In [49]:
# ↓ твой код ↓
class Tracked:
    def __init__(self):
        self.history = []

    def __setattr__(self, name, value):
        if name != "history":
            self.history.append(name)
        object.__setattr__(self, name, value)


In [50]:
# Проверка. Запускай после того, как напишешь решение.
t = Tracked()
t.a = 1
t.b = 2
t.a = 3

check("история по порядку", lambda: eq(t.history, ["a", "b", "a"]))
check("history не попал в историю", lambda: eq("history" in t.history, False))
check("значения читаются", lambda: eq((t.a, t.b), (3, 2)))

✓ история по порядку
✓ history не попал в историю
✓ значения читаются


### Задача 8. Почему `+=` и `=` дают разный результат

В задаче 3 `c1.items += ["a"]` изменил список у всех экземпляров, а `c1.items = c1.items + ["a"]`
создал бы личный. Объясни своими словами, почему поведение разное, и что именно ищет
интерпретатор в каждом из случаев.

Сформулируй ответ своими словами, без кода и без запуска.

In [ ]:
Потому что c1.items по умолчанию присваивает именно в аттрибут экземпляра.
Но при вызове c1.items += он смотрит, что у c1 нет собственного items, поэтому идёт на уровень класса, мутирует список (поэтому он и обновляется у всех объектов), а затем возвращает ссылку на этот список, которая присваивается в c1.items.
И теперь у c1 есть свой items, но это всё ещё ссылка на список класса.

### Задача 9. `__getattr__`, который врёт про отсутствие

`Proxy` отдаёт значения из словаря `data`. Сломано: `hasattr(p, "missing")` возвращает
`True`, а `getattr(p, "missing", None)` падает с `KeyError`.

Привести к контракту: неизвестное имя должно давать `AttributeError`, известное — значение.

In [56]:
# СЛОМАНО. Правь эту ячейку, пока проверка не станет зелёной.
class Proxy:
    def __init__(self):
        self.data = {"a": 1}

    def __getattr__(self, name):
        if name in self.data:
            return self.data[name]
        raise AttributeError

In [57]:
# Проверка. Запускай после того, как напишешь решение.
p = Proxy()
print(hasattr(p, "missing"))
check("известный ключ", lambda: eq(p.a, 1))
check("неизвестный — AttributeError", lambda: expect(AttributeError, lambda: p.missing))
check("hasattr не врёт", lambda: eq(hasattr(p, "missing"), False))

False
✓ известный ключ
✓ неизвестный — AttributeError
✓ hasattr не врёт
